# 18 · Ablation and contribution checks

## Purpose

Test the proposed combination as an empirical ablation of standard components, without claiming global novelty.

## Method

Compare probability-ranking, calibration, cost-sensitive selection, the combined calibrated cost strategy, and uncertainty-first review on the decision-study holdout; report AUC, PR-AUC, Brier, log loss, precision/recall@K, cost, and runtime.

## Interpretation

Use the displayed tables, figures, and saved metrics produced by this run. Do not fill this section with assumed outcomes. For every result, distinguish observed evidence from interpretation.

## Limitations

One holdout does not prove stability. Cost-sensitive ranking can be monotonic in probability for fixed costs, so equal rankings are a possible and informative result.

## References

Related established components and the novelty boundary are summarized in docs/references.md.


In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd().resolve()
while not (ROOT / 'returnwise').is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from IPython.display import display
from returnwise.research_state import load_state, save_state
state = load_state()
print('Project:', ROOT)
print('Notebook:', '18 · Ablation and contribution checks')


In [ ]:
from returnwise.experiments import ablation_table
from returnwise.research_state import load_state, save_state
import matplotlib.pyplot as plt

state = load_state()
predictions = state['decision_predictions']
predictions.attrs['selected_calibrator'] = state['decision_calibrator_name']
ablation = ablation_table(predictions, costs=state['cost_scenarios']['medium'])
display(ablation)
if 'expected_decision_cost' in ablation:
    plot_data = ablation[ablation['capacity_percent'] == 10]
    ax = plot_data.plot.bar(x='experiment', y='expected_decision_cost', figsize=(9, 4), legend=False, color='#126E63', title='Ablation expected cost at 10% review capacity')
    ax.set_ylabel('Simulated cost units')
    plt.tight_layout()
    plt.show()
state['ablation'] = ablation
from returnwise.experiments import log_experiment
log_experiment('Ablation study', None, {'results': ablation.to_dict(orient='records')},
               {'costs_are_simulated': True, 'decision_holdout_rows': len(predictions)})
save_state(state)
